# DistilBERT — 256 token eğitimi

Bu notebook, Drive'da önceden hazırlanmış WELFake v2 verisiyle çalışır. Hücreleri sırayla çalıştırın; Runtime > Change runtime type üzerinden GPU seçin. Model Colab'a indirilir.

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PROJECT = Path('/content/drive/MyDrive/fake_news_detection')
DRIVE_DATA = DRIVE_PROJECT / 'data' / 'prepared' / 'welfake_temiz_v2_seed42'
OUTPUT_ROOT = DRIVE_PROJECT / 'artifacts_v2' / 'deneyler'
CHECKPOINT_ROOT = DRIVE_PROJECT / 'artifacts_v2' / 'checkpointler'
print('Temiz veri klasörü:', DRIVE_DATA)

In [ ]:
import subprocess
import sys

REPO_URL = 'https://github.com/burak-dursunn/fake-news-detection.git'
PROJECT_DIR = Path('/content/fake-news-detection')
if (PROJECT_DIR / '.git').is_dir():
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only'], check=True)
elif not PROJECT_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_DIR)], check=True)
else:
    raise RuntimeError(f'Git deposu olmayan klasör mevcut: {PROJECT_DIR}')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT_DIR / 'requirements.txt')], check=True)

In [ ]:
import json
import shutil

LOCAL_DATA = Path('/content/prepared_data_v2')
required = ['welfake_temiz.csv', 'welfake_split_seed42.csv', 'welfake_temizleme_raporu.json', 'welfake_split_ozeti.json']
missing = [name for name in required if not (DRIVE_DATA / name).is_file()]
if missing:
    raise FileNotFoundError(f'Drive üzerinde v2 temiz veri eksik: {missing}\nKontrol edilen klasör: {DRIVE_DATA}')
report = json.loads((DRIVE_DATA / 'welfake_split_ozeti.json').read_text(encoding='utf-8'))
audit = report.get('audit', {})
if audit.get('identical_text_groups_across_splits') != 0 or audit.get('same_300_character_prefix_groups_across_splits') != 0:
    raise ValueError('v2 veri sızıntısı kontrolü eksik veya başarısız. Veri dosyalarını kontrol edin.')
LOCAL_DATA.mkdir(parents=True, exist_ok=True)
for name in required:
    shutil.copy2(DRIVE_DATA / name, LOCAL_DATA / name)
print('Temiz veri kopyalandı. Bölmeler:', report['split_counts'])
print('Sızıntı kontrolü:', audit)

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('GPU bulunamadı. Runtime menüsünden GPU seçin ve hücreleri tekrar çalıştırın.')
print('Kullanılacak GPU:', torch.cuda.get_device_name(0))

## DistilBERT eğitimini başlat
Çıktılar doğrudan Drive'a kaydedilir. Aynı adlı tamamlanmış deney varsa betik üzerine yazmaz. Çalışma zamanı kesilirse checkpointler Drive'da kalır; mevcut betik otomatik devam ettirmez.

In [ ]:
CONFIG = PROJECT_DIR / 'configs' / '01_DistilBERT_256Token.json'
EXPERIMENT_NAME = '01_DistilBERT_256Token'
if not CONFIG.is_file():
    raise FileNotFoundError(f'Deney ayar dosyası bulunamadı: {CONFIG}')
if (OUTPUT_ROOT / EXPERIMENT_NAME).exists():
    raise FileExistsError(f'Önceki deney klasörü mevcut, korunuyor: {OUTPUT_ROOT / EXPERIMENT_NAME}')

subprocess.run([
    sys.executable, '-m', 'src.run_experiment',
    '--config', str(CONFIG),
    '--data-dir', str(LOCAL_DATA),
    '--output-root', str(OUTPUT_ROOT),
    '--checkpoint-root', str(CHECKPOINT_ROOT),
], cwd=PROJECT_DIR, check=True)
print('Sonuçlar Drive klasörüne kaydedildi:', OUTPUT_ROOT / EXPERIMENT_NAME)

In [ ]:
from IPython.display import Image, display
result_dir = OUTPUT_ROOT / EXPERIMENT_NAME
for path in sorted(result_dir.glob('*.png')):
    print(path.name)
    display(Image(filename=str(path)))